# 02 · Retrieval and tools — solutions

**Prerequisites:** Sets, sorting, and functions; notebook 01 concepts are useful but not required.

**Learning objectives:** Implement deterministic retrieval; preserve provenance; expose a local tool; diagnose missing facts.

**Guide companion:** sections 6 in `LANGCHAIN_LANGGRAPH_LEARNING_GUIDE.md` at the project root.

**How to work:** Run setup, implement each challenge, then run its acceptance cell. Starter functions deliberately raise `NotImplementedError`; this is expected until you complete them. Restart the kernel and run all cells after finishing. You do not need any other notebook or paid API calls. Budget about 45–90 minutes, or longer for the capstone.

This copy contains complete implementations and answer explanations. Try the exercise notebook first.


In [ ]:
import os
os.environ["LANGSMITH_TRACING"] = "false"

# Fictional test data, not real people, policies, or research sources.
NOTES = [
    {"id": "s1", "url": "fixture://architecture", "text": "Cedar uses LangGraph to route research tasks."},
    {"id": "s2", "url": "fixture://review", "text": "Cedar pauses its workflow for human review."},
    {"id": "s3", "url": "fixture://ownership", "text": "Mira maintains Cedar."},
    {"id": "s4", "url": "fixture://team", "text": "Mira works on team Atlas."},
    {"id": "s5", "url": "fixture://policy", "text": "Atlas reviews Cedar evidence monthly."},
]
BY_ID = {note["id"]: note for note in NOTES}

import re
from langchain.tools import tool
STOP = {"a", "an", "the", "is", "are", "who", "what", "which", "how", "does", "do", "to", "of", "for", "on", "its"}

def tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP


## Challenge 1 · Rank the notes

Implement `search_notes(query, k=3)`. Score each note by the number of unique shared tokens using the supplied tokenizer. Exclude zero-score notes, sort by descending score then ascending source ID, and return at most `k` copies of note dictionaries. Preserve `id`, `url`, and `text`.
Reject non-integer `k` (including booleans) or values outside 1–5 with `ValueError`. Empty or stopword-only queries return an empty list.


In [ ]:
def search_notes(query: str, k: int = 3) -> list[dict]:
    if type(k) is not int or not 1 <= k <= 5:
        raise ValueError("k must be an integer from 1 to 5")
    terms = tokens(query)
    scored = [(len(terms & tokens(note["text"])), note) for note in NOTES]
    scored.sort(key=lambda item: (-item[0], item[1]["id"]))
    return [dict(note) for score, note in scored if score > 0][:k]


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
assert search_notes("Who maintains Cedar?")[0]["id"] == "s3"
assert [n["id"] for n in search_notes("Cedar")] == ["s1", "s2", "s3"], "Break equal scores by source ID"
assert search_notes("quasar") == []
assert search_notes("") == [] and search_notes("the who is") == []
assert len(search_notes("Cedar", k=1)) == 1
assert [n["id"] for n in search_notes("MIRA maintains CEDAR!!!", 1)] == ["s3"]
for bad_k in [0, 6, -1, True, 2.5]:
    try:
        search_notes("Cedar", bad_k)
    except ValueError:
        pass
    else:
        raise AssertionError(f"Reject invalid k: {bad_k}")
copy = search_notes("maintains", 1)[0]
assert set(copy) == {"id", "url", "text"}
copy["text"] = "changed"
assert BY_ID["s3"]["text"] == "Mira maintains Cedar.", "Return copies, not shared dictionaries"
print("PASS: ranking, provenance, empty queries, and input limits")


### Why this works

Tie-breaking makes repeated runs predictable. Returning copies keeps accidental edits from corrupting the shared fixture. A nonempty result establishes lexical overlap, not that the question can be answered.


## Challenge 2 · Expose retrieval as a tool

Implement `build_lookup_tool()` returning a LangChain tool named `lookup_notes`. It must accept one string argument named `query`, have a meaningful description, and return `search_notes(query)`.
Do not create an agent or call a model. You will invoke the tool directly.


In [ ]:
def build_lookup_tool():
    @tool
    def lookup_notes(query: str) -> list[dict]:
        """Search fictional Cedar project notes and return IDs, locations, and text."""
        return search_notes(query)
    return lookup_notes


**Acceptance check:** run after implementing the cell above. An unfinished starter intentionally fails; an unexpected assertion failure describes behavior to fix.


In [ ]:
lookup = build_lookup_tool()
assert lookup.name == "lookup_notes"
assert lookup.description.strip()
assert lookup.invoke({"query": "Who maintains Cedar?"}) == search_notes("Who maintains Cedar?")
assert lookup.invoke({"query": "quasar"}) == []
schema = lookup.args_schema.model_json_schema()
assert schema["properties"]["query"]["type"] == "string"
print("PASS: tool interface; no model needed")


### Why this works

The tool packages a capability for a caller. A model may select it in an agent, but the function itself remains ordinary Python. Tests of tool behavior can run without a model.


## Investigation · Retrieval is not an answer

Run the probe below, then answer the reflection questions. It intentionally has no universal relevance score: you must inspect the evidence.


In [ ]:
for query in ["Who maintains Cedar?", "Who is responsible for Cedar?", "What is Cedar's budget?"]:
    print(query)
    print([(note["id"], note["text"]) for note in search_notes(query)])


## Reflection

1. Which query retrieves notes while lacking the requested fact?
2. Why can the paraphrased ownership query rank differently?
3. Propose one retrieval improvement and one evaluation case that would reveal whether it helps.


### Discussion

The budget question retrieves Cedar notes but none states a budget. “Responsible” does not match “maintains,” so simple word overlap loses useful intent. Query expansion, vector search, or hybrid retrieval might improve paraphrases; compare results on held-out paraphrases and unsupported questions before adopting them.


## References

- [Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [Retrieval](https://docs.langchain.com/oss/python/deepagents/retrieval)
